# Create the Corpus Problem Lists

The notebook below creates the lists of documents for the corpus and datatype. This is used for iterating across the documents in the corpus when creating jobscripts that pull the known and unknown documents. To run this for a new set of data simply change the **corpus** and **data_type** parameters.

In [67]:
import sys
import pandas as pd

from from_root import from_root

sys.path.insert(0, str(from_root("src")))

from utils import get_base_location, build_metadata_df, apply_temp_doc_id
from read_and_write_docs import read_jsonl, read_rds

In [68]:
corpus      = "ACL"
data_type   = "test"

# Set NAS so can run on Windows laptop seamlessly
nas_base_loc = get_base_location()

# --- set your args (strings) ---
nas_base_loc = "/Users/user/Documents/uni_work_offline"

known_loc = f"{nas_base_loc}/datasets/author_verification/{data_type}/{corpus}/known_raw.jsonl"
unknown_loc = f"{nas_base_loc}/datasets/author_verification/{data_type}/{corpus}/unknown_raw.jsonl"
metadata_loc = f"{nas_base_loc}/datasets/author_verification/{data_type}/metadata.rds"

save_loc = f"{nas_base_loc}/datasets/author_verification/{data_type}/{corpus}"

## Read Data

In [69]:
metadata = read_rds(metadata_loc)
filtered_metadata = metadata[metadata['corpus'] == corpus]

known = read_jsonl(known_loc)
unknown = read_jsonl(unknown_loc)

In [70]:
filtered_metadata

,problem,corpus,known_author,unknown_author
0,"Ibekwe-SanJuan, Fidelia vs Ibekwe-SanJuan, Fid...",ACL,"Ibekwe-SanJuan, Fidelia","Ibekwe-SanJuan, Fidelia"
1,"Ibekwe-SanJuan, Fidelia vs Ide, Nancy",ACL,"Ibekwe-SanJuan, Fidelia","Ide, Nancy"
2,"Ide, Nancy vs Ide, Nancy",ACL,"Ide, Nancy","Ide, Nancy"
3,"Ide, Nancy vs Issac, Fabrice",ACL,"Ide, Nancy","Issac, Fabrice"
4,"Issac, Fabrice vs Issac, Fabrice",ACL,"Issac, Fabrice","Issac, Fabrice"
...,...,...,...,...
275,"Zeman, Daniel vs Zhai, ChengXiang",ACL,"Zeman, Daniel","Zhai, ChengXiang"
276,"Zhai, ChengXiang vs Zhai, ChengXiang",ACL,"Zhai, ChengXiang","Zhai, ChengXiang"
277,"Zhai, ChengXiang vs Zuber, Richard",ACL,"Zhai, ChengXiang","Zuber, Richard"
278,"Zuber, Richard vs Agerri, Rodrigo",ACL,"Zuber, Richard","Agerri, Rodrigo"


## Create Agg Problem Level Metadata

In [71]:
agg_problems = filtered_metadata['problem'].drop_duplicates()

## Create Metadata

Quite a convoluted process.

In [72]:
# Build the dataframe
complete_metadata = build_metadata_df(filtered_metadata, known, unknown)

# Set blank text column for function to work
complete_metadata['text'] = ''

# Rename the known column and create the new doc_id
complete_metadata.rename(columns={"known_doc_id": "orig_doc_id"}, inplace=True)
complete_metadata = apply_temp_doc_id(complete_metadata)
complete_metadata.rename(columns={
    "orig_doc_id": "orig_known_doc_id",
    "doc_id": "known_doc_id",
    "unknown_doc_id": "orig_doc_id"
}, inplace=True)

# Do the same for the unknown
complete_metadata = apply_temp_doc_id(complete_metadata)
complete_metadata.rename(columns={
    "orig_doc_id": "orig_unknown_doc_id",
    "doc_id": "unknown_doc_id",
}, inplace=True)

# Sort columns
complete_metadata = complete_metadata[["sample_id", "problem", "corpus", "known_doc_id", "unknown_doc_id"]]

## View the data

In [73]:
complete_metadata.head()

,sample_id,problem,corpus,known_doc_id,unknown_doc_id
0,1,"Ibekwe-SanJuan, Fidelia vs Ibekwe-SanJuan, Fid...",ACL,ibekwe_sanjuan_fidelia_678_pdf_txt_elra_2006,ibekwe_sanjuan_fidelia_p98_1092_acl_1998
1,2,"Ibekwe-SanJuan, Fidelia vs Ide, Nancy",ACL,ibekwe_sanjuan_fidelia_678_pdf_txt_elra_2006,ide_nancy_w00_1506_iccl_2000
2,3,"Ide, Nancy vs Ide, Nancy",ACL,ide_nancy_w14_3005_acl_2014,ide_nancy_w00_1506_iccl_2000
3,4,"Ide, Nancy vs Issac, Fabrice",ACL,ide_nancy_w14_3005_acl_2014,issac_fabrice_c10_2056_coc_2010
4,5,"Issac, Fabrice vs Issac, Fabrice",ACL,issac_fabrice_w98_0118_ircs_1998,issac_fabrice_c10_2056_coc_2010


## Create the Document Lists

In [74]:
known_doc_list = pd.Series(complete_metadata["known_doc_id"].astype(str))
unknown_doc_list = pd.Series(complete_metadata["unknown_doc_id"].astype(str))
problem_doc_list = known_doc_list + ' vs ' + unknown_doc_list

## Get Number of Rows in the Dataset

This is used for the jobscript.

In [75]:
num_rows_for_jobscript = complete_metadata.shape[0]
print(f"Number of rows needed in jobscript: {num_rows_for_jobscript}")

Number of rows needed in jobscript: 280


## Save the Lists

In [76]:
print(f"Writing agg problem list - {len(agg_problems)} Aggregated Problems")
agg_problems.to_csv(f"{save_loc}/agg_problem_list.txt", index=False, header=False)
print(f"Writing problem doc list - {len(problem_doc_list)} Problems")
problem_doc_list.to_csv(f"{save_loc}/problem_doc_list.txt", index=False, header=False)
print(f"Writing known doc list - {len(known_doc_list)} Documents")
known_doc_list.to_csv(f"{save_loc}/known_doc_list.txt", index=False, header=False)
print(f"Writing unknown doc list - {len(unknown_doc_list)} Documents")
unknown_doc_list.to_csv(f"{save_loc}/unknown_doc_list.txt", index=False, header=False)
print("Wrote doc lists")

Writing agg problem list - 280 Aggregated Problems
Writing problem doc list - 280 Problems
Writing known doc list - 280 Documents
Writing unknown doc list - 280 Documents
Wrote doc lists
